In [1]:
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.utils import ImageReader
import os
import io
import pandas as pd
import numpy as np
# import mne
import json

from utils import *
from eeg_qc import compute_eeg_pipeline, test_eeg_pipeline
from ecg_qc import ecg_qc 
from eda_qc import eda_qc
from rsp_qc import *
from mic_qc import *
from lsl_problem import *
from et_qc import *
from webcam_qc import *
from behavior_qc import *
from generate_csv import *
import seaborn as sns
# import matplotlib

## Load data from QC csv file and data dictionary

In [ ]:
filename = 'CUNY_QC_old.csv'
dataset = pd.read_csv(filename)
with open("data_dict.json", "r") as f:
    metric_data = json.load(f)

## Create dataframe with mean and std. deviation for all QC measures 

In [3]:
quant_metrics = dataset.select_dtypes(include=['float64', 'int64']).columns.tolist()
number_of_subjects = len(dataset['Subject'])

new_df = pd.DataFrame(columns = ['Metric', 'Mean, Std. Deviation'])
new_df['Metric'] = [metric_data[metric]['Print Name'] for metric in quant_metrics]
for metric in quant_metrics:
    mean_value = np.mean(dataset[metric].dropna().tolist())
    std_value = np.std(dataset[metric].dropna().tolist())
    new_df.loc[new_df['Metric'] == metric_data[metric]['Print Name'], 'Mean, Std. Deviation'] = f"M = {round(mean_value,3)} {metric_data[metric]['Unit']}\nSD = {round(std_value,3)}"
new_df['Metric'] = new_df['Metric'].str.wrap(30)
print(new_df)

                                               Metric  \
0                              Duration of experiment   
1                         Duration of impedence check   
2   Average response time across\nall story listen...   
3               Percent Good before artifact\nremoval   
4                             Effective sampling rate   
..                                                ...   
72  Percent of Physiology stream\nduration compare...   
73                 Duration of EEG stream in\nseconds   
74  Percent of EEG stream duration\ncompared to ex...   
75                       Expected Duration in seconds   
76                       Percent of Expected Duration   

            Mean, Std. Deviation  
0    M = 2129.364 s\nSD = 74.792  
1     M = 281.364 s\nSD = 20.781  
2      M = 24.273 s\nSD = 29.809  
3        M = 88.093 %\nSD = 4.45  
4   M = 44098.965 Hz\nSD = 0.568  
..                           ...  
72      M = 99.999 %\nSD = 0.004  
73  M = 2189.566 s\nSD = 220.981  


## Create dataset/sample report with distibution plots for all QC measures

In [4]:
def make_distplot(metric):
    #print(metric.columns[0])
    metric = metric.rename(columns={metric.columns[0]:''})
    fig, ax = plt.subplots(figsize=(10, 2))
    sns.set_theme(style="white")
    sns.violinplot(metric, fill=False, inner='box', linewidth=2, split=True, inner_kws=dict(box_width=10, whis_width=2, color='indianred'), orient='h')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    fig.show()
    buf.seek(0)
    #plt.show()
    plt.close(fig)

    return buf

In [5]:
# PDF structure
parent_folder = os.getcwd() + '/'
pdf_path = f"{parent_folder}Dataset_Report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4)
elements = []
styles = getSampleStyleSheet()
#styleN = styles["BodyText"]
styleN = styles["Normal"]
styleN.wordWrap = "CJK"


# Define subtitle style if not already done
subtitle_style = ParagraphStyle(
    name="Subtitle",
    parent=styles["Heading2"],
    fontSize=14,
    leading=16,
    textColor="gray",
    spaceAfter=12,
    alignment=1  # Centered
)

# page number function
def add_page_number(canvas, doc):
    page_num = f'{canvas.getPageNumber()}'
    canvas.setFont("Helvetica", 9)
    canvas.drawRightString(570, 20, page_num)

elements.append(Paragraph(f"Dataset Report", styles["Title"]))
elements.append(Paragraph(f"Collection Period: {dataset['Collection Date'][0].split(' ')[0]} - {dataset['Collection Date'][len(dataset['Collection Date'])-1].split(' ')[0]}, N= {number_of_subjects}", subtitle_style))
elements.append(Spacer(1, 12))

metric_data_keys = [key for key in quant_metrics]#list(metric_data.keys())
new_df_with_plots = new_df.copy()
new_df_with_plots["Distribution Plot"] = None

for idx, row in new_df_with_plots.iterrows():
    #buffer = make_distplot(dataset[['Metric']])
    buffer = make_distplot(dataset[[metric_data_keys[idx]]])
    image = ImageReader(buffer)
    orig_width, orig_height = image.getSize()
    img = Image(buffer, width=orig_width/5, height=orig_height/3)
    new_df_with_plots.at[idx, "Distribution Plot"] = img
data_list = [new_df_with_plots.columns.tolist()] + new_df_with_plots.values.tolist() 
#tableWidth = 500
#dftable = Table(data_list, colWidths=tableWidth/3, repeatRows=1)
dftable = Table(data_list, repeatRows=1)
dftable.hAlign = 'CENTER'



dftable.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.black),
    ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
]))

elements.append(dftable) 
elements.append(Spacer(1, 12))

doc.build(elements, onFirstPage = add_page_number, onLaterPages = add_page_number)
print(f'PDF created: {pdf_path}')

PDF created: /Users/apurva.gokhe/Documents/GitHub/MOBI_QC/src/MOBI_QC/important_files/Dataset_Report.pdf
